# 📘 Aula 06c: Classificação Multiclasse de Texto com Reuters (Keras & PyTorch)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula06c_reuters.ipynb)

Nesta aula prática avançada, entramos no universo do **Processamento de Linguagem Natural (NLP)** aplicado à **classificação de tópicos jornalísticos**. 

Trabalharemos com o clássico dataset de notícias da agência **Reuters**, composto por milhares de reportagens curtas classificadas em **46 temas econômicos e noticiosos distintos**. Este é um problema de **classificação multiclasse de alta dimensionalidade**, que nos permitirá dominar técnicas de vetorização de texto (*Bag of Words*), diagnóstico de *overfitting* via curvas de perda/validação, dimensionamento de camadas ocultas (evitando gargalos de informação) e a comparação direta entre **Keras** e **PyTorch**.

---

### 🎯 Objetivo da Aula
1. Compreender a representação de textos como sequências numéricas de índices e sua decodificação reversa para palavras legíveis.
2. Implementar a vetorização *Bag of Words* binária tanto **artesanalmente com NumPy** quanto via utilitários do **Keras** (`Tokenizer`).
3. Projetar e treinar uma rede neural densa multiclasse para 46 categorias no Keras com `Categorical Crossentropy` e ativação `Softmax`.
4. Monitorar métricas de treino vs validação para identificar o ponto exato de início do *overfitting* (e aplicar parada antecipada prática).
5. Analisar o fenômeno do **gargalo de informação (*information bottleneck*)** reduzindo neurônios ocultos.
6. Reimplementar todo o pipeline de treinamento em tensores puros e laço de gradiente manual com **PyTorch**.

---

### 💡 A Intenção Pedagógica do Professor
* **Construção do Zero vs Biblioteca**: O professor constrói primeiro a função manual `vectorize_sequence` em NumPy para demonstrar exatamente como textos viram tensores binários no computador, antes de chamar o `Tokenizer` do Keras.
* **Diagnóstico Visual de Overfitting**: Mostrar ao estudante que treinar por muitas épocas não significa melhorar o modelo: as curvas de perda revelam quando a rede começa a apenas memorizar o treino.
* **Gargalo de Informação**: Demonstrar experimentalmente por que uma camada oculta com menos neurônios (ex: 32) do que o número de classes de saída (46) perde capacidade de representar os dados.
* **Dualidade Keras vs PyTorch**: Treinar o mesmo modelo nos dois frameworks para fixar a compreensão conceitual independente de sintaxe.

---

### 📚 Glossário da Aula
* **Reuters Dataset**: Coleção de 11.228 notícias publicadas pela agência Reuters em 1986, categorizadas em 46 tópicos mútuos.
* **Bag of Words (BoW) Binário**: Vetor de presença/ausência de palavras. Cada posição representa um termo do vocabulário ($1$ se a palavra ocorre no texto, $0$ caso contrário).
* **Information Bottleneck (Gargalo de Informação)**: Quando uma camada oculta possui dimensão menor do que a complexidade necessária para separar as classes de saída, forçando perda irreversível de informação.
* **Overfitting (Sobreajuste)**: Momento em que o erro de treino continua caindo, mas o erro de validação volta a subir, indicando perda de generalização.

---

### 🧭 Trilha da Aula
1. [Carregamento do Dataset Reuters e Exploração](#1-carregamento-do-dataset-reuters)
2. [Decodificação de Sequências de Índices em Texto Legível](#2-decodificação-de-palavras)
3. [Vetorização de Texto: NumPy Manual vs Keras Tokenizer](#3-vetorização-de-texto-bag-of-words)
4. [Codificação dos Rótulos em One-Hot (46 Classes)](#4-codificação-dos-rótulos)
5. [Construção do Modelo Sequencial no Keras](#5-construção-do-modelo-keras)
6. [Treinamento e Visualização de Curvas (Overfitting)](#6-treinamento-e-curvas-de-perda)
7. [Ajuste de Épocas & Avaliação no Teste](#7-ajuste-de-épocas-e-avaliação)
8. [Análise de Gargalo de Informação (Camadas com 64 e 32 Neurônios)](#8-gargalo-de-informação)
9. [Reimplementação Completa em PyTorch](#9-reimplementação-em-pytorch)


## 1. Carregamento do Dataset Reuters

**O que estamos fazendo**: Carregar o acervo de notícias do Reuters já pré-indexado pelo Keras, limitando o vocabulário às 10.000 palavras mais frequentes.

**Por que fazer assim**: Palavras extremamente raras geram ruído e aumentam o tamanho da matriz sem agregar sinal preditivo consistente para a rede.

> [!NOTE]
> O parâmetro `test_split=0.2` separa 20% das notícias para o conjunto final de teste independente.

> [!IMPORTANT]
> Cada notícia é originalmente uma lista de números inteiros onde cada número é o índice de uma palavra no dicionário.



In [ ]:
# Importa o módulo Keras do TensorFlow
from tensorflow import keras
# Importa o dataset pré-processado Reuters
from tensorflow.keras.datasets import reuters
# Importa métrica de acurácia do Scikit-Learn
from sklearn.metrics import accuracy_score
# Importa NumPy para manipulação matricial
import numpy as np
# Importa Matplotlib para visualização gráfica das curvas de perda e acurácia
import matplotlib.pyplot as plt

# Carrega os dados limitando ao vocabulário das 10.000 palavras mais comuns
# Divisão: 80% treino (train_data, train_labels) e 20% teste (test_data, test_labels)
(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000, test_split=0.2)


## 2. Inspeção da Quantidade de Amostras

**O que estamos fazendo**: Verificar quantas notícias estão disponíveis nas partições de treino e teste.

**Por que fazer assim**: Fundamental para confirmar as proporções dos conjuntos e planejar o particionamento de validação.



In [ ]:
# Exibe o total de notícias destinadas ao treino (8.982 notícias)
print(f'Total de amostras de treino: {len(train_data)}')
# Exibe o total de notícias destinadas ao teste (2.246 notícias)
print(f'Total de amostras de teste: {len(test_data)}')


## 3. Estrutura Interna das Notícias

**O que estamos fazendo**: Inspecionar as duas primeiras notícias em seu formato cru numérico.

**Por que fazer assim**: Observe que cada texto já foi tokenizado: cada número inteiro representa um termo específico do vocabulário.



In [ ]:
# Visualiza as duas primeiras sequências de inteiros do conjunto de treino
print('Primeira notícia (índices numéricos):', train_data[0][:20], '...')
print('Segunda notícia (índices numéricos):', train_data[1][:20], '...')


## 4. Variação no Comprimento dos Textos

**O que estamos fazendo**: Verificar a contagem de palavras das notícias.

**Por que fazer assim**: Notícias reais possuem tamanhos variados. Como redes neurais densas exigem vetores de tamanho fixo na entrada, precisaremos de vetorização.

> [!TIP]
> Nas camadas densas, a entrada precisa ter tamanho padronizado; na vetorização Bag of Words, o vetor sempre terá tamanho 10.000 independente do tamanho do texto!



In [ ]:
# Quantidade de palavras contidas na primeira notícia
print(f'Comprimento da notícia 0: {len(train_data[0])} palavras')
# Quantidade de palavras contidas na segunda notícia
print(f'Comprimento da notícia 1: {len(train_data[1])} palavras')


## 5. Decodificação de Índices para Texto Legível

**O que estamos fazendo**: Construir uma função que inverte o dicionário de palavras e reconstrói o texto em inglês.

**Por que fazer assim**: Permite auditar qualitativamente os dados e entender o que a rede está processando.

> [!NOTE]
> Os índices 0, 1 e 2 são reservados no Keras para 'padding', 'início de sequência' e 'palavra desconhecida'. Por isso aplica-se o deslocamento `i - 3`.



In [ ]:
def decode_words(word_index, sentence):
  """
  Decodifica uma lista de inteiros em uma frase de texto legível.
  word_index: dicionário mapeando palavra -> inteiro.
  sentence: lista de inteiros representando a notícia.
  """
  # Inverte o dicionário: cria mapeamento de inteiro -> palavra
  reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])
  # Reconstrói a frase substituindo os índices pelas palavras correspondentes (com offset de 3)
  decoded_sentence = ' '.join([reverse_word_index.get(i-3, '?') for i in sentence])
  return decoded_sentence

# Decodifica e imprime o texto completo da primeira notícia de treino
print('Notícia 0 decodificada:')
print(decode_words(reuters.get_word_index(), train_data[0]))


## 6. Decodificação da Segunda Notícia

**O que estamos fazendo**: Decodificar a segunda notícia para verificar a diversidade do conteúdo jornalístico financeiro.

**Por que fazer assim**: Permite confirmar a consistência do dicionário de decodificação.



In [ ]:
# Decodifica e imprime a segunda notícia de treino
print('Notícia 1 decodificada:')
print(decode_words(reuters.get_word_index(), train_data[1]))


## 7. Verificação do Número de Classes (Tópicos)

**O que estamos fazendo**: Contar a quantidade de tópicos distintos presentes nos rótulos de treino.

**Por que fazer assim**: Saber o número exato de classes ($46$) define o número de neurônios da camada final da rede neural.

> [!IMPORTANT]
> Como existem 46 classes possíveis, trata-se de um problema multiclasse com muitas saídas, muito além de uma classificação binária simples!



In [ ]:
# Conta a quantidade de classes únicas presentes no dataset
num_classes = len(set(train_labels))
print(f'Total de classes/tópicos distintos: {num_classes}')


## 8. Vetorização Artesanal em NumPy (Bag of Words)

**O que estamos fazendo**: Implementar manualmente uma função de vetorização binária que transforma cada notícia em um vetor esparso de dimensão 10.000.

**Por que fazer assim**: O professor faz isso do zero para mostrar a essência algorítmica: uma matriz de zeros onde marcamos $1$ na coluna correspondente a cada palavra presente.

> [!NOTE]
> Uma notícia com as palavras [3, 5] terá a linha preenchida com zeros, exceto nas colunas 3 e 5 que receberão o valor 1.0.



In [ ]:
def vectorize_sequence(sequence, dimension=10000):
  """
  Converte uma lista de sequências numéricas em uma matriz binária (One-Hot por palavra).
  sequence: lista de listas de inteiros.
  dimension: tamanho fixo do vocabulário (10.000).
  """
  # Inicializa uma matriz de zeros com formato [número_de_amostras, dimensão_do_vocabulário]
  results = np.zeros((len(sequence), dimension))
  # Para cada notícia, marca 1.0 nas colunas relativas às palavras presentes
  for i, seq in enumerate(sequence):
    results[i, seq] = 1.
  return results

# Vetoriza os conjuntos de treino e teste artesanalmente
X_train = vectorize_sequence(train_data)
X_test = vectorize_sequence(test_data)
print('Formato de X_train:', X_train.shape)
print('Formato de X_test:', X_test.shape)
print('Primeiras 48 posições das 3 primeiras amostras:
', X_train[:3, :48])


## 9. Vetorização Equivalente com o Tokenizer do Keras

**O que estamos fazendo**: Utilizar a classe `Tokenizer` do Keras para realizar a mesma transformação matricial.

**Por que fazer assim**: Demonstra a transição do código artesanal em NumPy para o padrão profissional das bibliotecas de Deep Learning.

> [!TIP]
> `tokenizer.sequences_to_matrix(..., mode='binary')` gera exatamente a mesma matriz binária que implementamos na célula anterior!



In [ ]:
# Importa o Tokenizer utilitário do Keras para processamento de texto
from tensorflow.keras.preprocessing.text import Tokenizer

# Instancia o Tokenizer configurado para as 10.000 palavras mais frequentes
tokenizer = Tokenizer(num_words=10000)
# Converte as sequências de inteiros diretamente na matriz binária Bag-of-Words
X_train = tokenizer.sequences_to_matrix(train_data)
X_test = tokenizer.sequences_to_matrix(test_data)
print('Formato X_train (Tokenizer):', X_train.shape)
print('Formato X_test (Tokenizer):', X_test.shape)
print('Amostra da matriz gerada:
', X_train[:3, :48])


## 10. Inspeção dos Rótulos Originais

**O que estamos fazendo**: Visualizar os primeiros rótulos numéricos do conjunto de treino.

**Por que fazer assim**: Os rótulos são números inteiros entre 0 e 45 representando a categoria temática da notícia.



In [ ]:
# Exibe os 10 primeiros rótulos inteiros de classes
print('Primeiros 10 rótulos de classe:', train_labels[:10])


## 11. Codificação One-Hot dos Rótulos (to_categorical)

**O que estamos fazendo**: Converter os rótulos inteiros em vetores One-Hot com dimensão 46.

**Por que fazer assim**: Para calcular a perda de entropia cruzada categórica com Softmax, a saída esperada deve ser uma distribuição com valor 1.0 na classe correta e 0.0 nas demais.



In [ ]:
# Utilitário do Keras para codificação One-Hot
from tensorflow.keras.utils import to_categorical

# Converte os vetores de inteiros em matrizes binárias [N, 46]
y_train = to_categorical(train_labels)
y_test = to_categorical(test_labels)
print('Formato de y_train após One-Hot:', y_train.shape)
print('Formato de y_test após One-Hot:', y_test.shape)


## 12. Construção da Rede Neural com Keras

**O que estamos fazendo**: Montar um modelo Sequencial com duas camadas ocultas de 64 neurônios com ativação ReLU e saída com 46 neurônios Softmax.

**Por que fazer assim**: Cada camada de 64 neurônios projeta o vetor de 10.000 palavras em um espaço denso. 64 dimensões é suficiente para não criar um gargalo para 46 classes.

> [!NOTE]
> Utilizamos o otimizador `RMSprop` e a perda `categorical_crossentropy` com monitoramento da métrica `accuracy`.



In [ ]:
# Importa camadas e modelo sequencial do Keras
from tensorflow import keras
from tensorflow.keras import layers

# Definição da arquitetura da rede
model = keras.Sequential([
  # Primeira camada oculta: 64 neurônios com ativação ReLU
  layers.Dense(64, activation='relu'),
  # Segunda camada oculta: 64 neurônios com ativação ReLU
  layers.Dense(64, activation='relu'),
  # Camada de saída: 46 neurônios com Softmax para gerar distribuição de probabilidades
  layers.Dense(46, activation='softmax')
])

# Compilação do modelo com RMSprop e entropia cruzada categórica
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])


## 13. Separação do Conjunto de Validação

**O que estamos fazendo**: Separar as primeiras 1.000 amostras do treino para criar um conjunto de validação independente.

**Por que fazer assim**: O conjunto de validação nos permite monitorar a perda e acurácia da rede em dados não vistos a cada época de treinamento, detectando overfitting sem tocar no conjunto de teste.



In [ ]:
# Tamanho do conjunto de validação
n_val = 1000

# Conjunto de validação (primeiras 1000 amostras)
X_val = X_train[:n_val]
y_val = y_train[:n_val]

# Conjunto de treino parcial restante (a partir da amostra 1000)
partial_X_train = X_train[n_val:]
partial_y_train = y_train[n_val:]

print(f'Treino parcial: {len(partial_X_train)} amostras | Validação: {len(X_val)} amostras')


## 14. Treinamento Inicial por 20 Épocas

**O que estamos fazendo**: Executar o treinamento da rede por 20 épocas com minibatches de 512 amostras, passando os dados de validação para monitoramento.

**Por que fazer assim**: Armazenamos o objeto `history` para inspecionar a evolução da função de perda e da acurácia época por época.



In [ ]:
# Treina o modelo e registra o histórico de métricas
history = model.fit(partial_X_train,
                    partial_y_train,
                    epochs=20,
                    batch_size=512,
                    validation_data=(X_val, y_val))


## 15. Diagnóstico de Perda (Loss): Treino vs Validação

**O que estamos fazendo**: Plotar o gráfico comparativo da perda de treino e de validação ao longo das 20 épocas.

**Por que fazer assim**: Observe o ponto de inflexão: a perda de treino cai continuamente, mas a perda de validação atinge o mínimo por volta da 9ª época e depois volta a subir!

> [!WARNING]
> Esse descolamento onde a perda de validação sobe enquanto o treino cai é a assinatura visual clássica do OVERFITTING (sobreajuste)!



In [ ]:
# Extrai as listas de valores de perda do histórico
history_dict = history.history
loss_values = history_dict['loss']
val_loss_values = history_dict['val_loss']
epochs = range(1, len(loss_values) + 1)

# Gráfico da perda
plt.figure(figsize=(8, 5))
plt.plot(epochs, loss_values, 'bo', label='Perda no Treino (Training loss)')
plt.plot(epochs, val_loss_values, 'b', label='Perda na Validação (Validation loss)')
plt.title('Perda de Treinamento e Validação (Reuters)')
plt.xlabel('Épocas')
plt.ylabel('Custo (Loss)')
plt.legend()
plt.grid(True)
plt.show()


## 16. Diagnóstico de Acurácia: Treino vs Validação

**O que estamos fazendo**: Plotar as curvas de acurácia no treino e na validação.

**Por que fazer assim**: Confirma o mesmo diagnóstico: a acurácia de validação estabiliza em torno de ~80% a 82% após 9-10 épocas, enquanto a acurácia de treino continua subindo artificialmente.



In [ ]:
# Extrai as métricas de acurácia do histórico
history_dict = history.history
acc_values = history_dict['accuracy']
val_acc_values = history_dict['val_accuracy']
epochs = range(1, len(acc_values) + 1)

# Gráfico da acurácia
plt.figure(figsize=(8, 5))
plt.plot(epochs, acc_values, 'bo', label='Acurácia no Treino (Training acc)')
plt.plot(epochs, val_acc_values, 'b', label='Acurácia na Validação (Validation acc)')
plt.title('Acurácia de Treinamento e Validação')
plt.xlabel('Épocas')
plt.ylabel('Acurácia')
plt.legend()
plt.grid(True)
plt.show()


## 17. Re-treinamento com Early Stopping Manual (10 Épocas)

**O que estamos fazendo**: Treinar o modelo por apenas 10 épocas para interromper o aprendizado exatamente no ponto ótimo antes do overfitting.

**Por que fazer assim**: Evita que os pesos fiquem excessivamente especializados no conjunto de treino, maximizando a generalização no conjunto de teste.



In [ ]:
# Recompila e treina por 10 épocas (ponto ótimo identificado nas curvas)
model = keras.Sequential([
  layers.Dense(64, activation='relu'),
  layers.Dense(64, activation='relu'),
  layers.Dense(46, activation='softmax')
])
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(partial_X_train,
                    partial_y_train,
                    epochs=10,
                    batch_size=512,
                    validation_data=(X_val, y_val))


## 18. Geração de Predições no Conjunto de Teste

**O que estamos fazendo**: Calcular as predições para todas as notícias de teste via `model.predict(X_test)`.

**Por que fazer assim**: Cada linha da saída é um vetor de 46 probabilidades estimadas pela função Softmax.



In [ ]:
# Gera as probabilidades para as 2.246 amostras de teste
y_pred = model.predict(X_test)
print('Formato da matriz de predições y_pred:', y_pred.shape)


## 19. Inspeção Numérica das Probabilidades

**O que estamos fazendo**: Inspecionar as probabilidades geradas para as duas primeiras notícias de teste.

**Por que fazer assim**: A soma de cada linha é igual a 1.0 (propriedade garantida pela Softmax).



In [ ]:
# Exibe as probabilidades das duas primeiras notícias de teste
print('Probabilidades estimadas para a amostra 0 (primeiras 10 classes):
', y_pred[0][:10])
print('Soma das probabilidades da amostra 0:', np.sum(y_pred[0]))


## 20. Comparação Visual: Classe Predita vs Classe Real

**O que estamos fazendo**: Comparar a classe mais provável predita (`argmax`) com o rótulo real para as 5 primeiras notícias de teste.

**Por que fazer assim**: Permite verificar visualmente se as decisões tomadas pela rede coincidem com as anotações humanas.



In [ ]:
# Compara os 5 primeiros casos
print('Predição vs Rótulo Real:')
for i in range(5):
  print(f'Amostra {i} -> Predito: {y_pred[i].argmax()} | Real: {y_test[i].argmax()}')


## 21. Avaliação Formal no Conjunto de Teste

**O que estamos fazendo**: Utilizar o método `model.evaluate` para calcular a perda e acurácia consolidadas no teste.

**Por que fazer assim**: O conjunto de teste é a métrica final e não tendenciosa do desempenho do modelo em produção.



In [ ]:
# Avalia nos dados de teste [Loss, Acurácia]
results = model.evaluate(X_test, y_test)
print(f'Teste -> Perda: {results[0]:.4f} | Acurácia: {results[1] * 100:.2f}%')


## 22. Avaliação Formal no Treino Parcial

**O que estamos fazendo**: Calcular a perda e acurácia no conjunto de treino parcial para comparação.

**Por que fazer assim**: Permite quantificar o gap entre treino e teste.



In [ ]:
# Avalia no treino parcial
results_train = model.evaluate(partial_X_train, partial_y_train)
print(f'Treino -> Perda: {results_train[0]:.4f} | Acurácia: {results_train[1] * 100:.2f}%')


## 23. Avaliação Formal na Validação

**O que estamos fazendo**: Calcular a perda e acurácia no conjunto de validação.

**Por que fazer assim**: Confirma a consistência entre o conjunto de validação e o conjunto de teste independente.



In [ ]:
# Avalia na validação
results_val = model.evaluate(X_val, y_val)
print(f'Validação -> Perda: {results_val[0]:.4f} | Acurácia: {results_val[1] * 100:.2f}%')


## 24. Testando uma Camada Oculta Única (64 Neurônios)

**O que estamos fazendo**: Treinar uma arquitetura mais simples com apenas uma camada oculta de 64 neurônios.

**Por que fazer assim**: O objetivo pedagógico é verificar se uma única camada é suficiente para separar os 46 tópicos jornalísticos.



In [ ]:
# Arquitetura com apenas 1 camada oculta de 64 neurônios
model = keras.Sequential([
  layers.Dense(64, activation='relu'),
  layers.Dense(46, activation='softmax')
])
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(partial_X_train,
                    partial_y_train,
                    epochs=20,
                    batch_size=512,
                    validation_data=(X_val, y_val),
                    verbose=0)
res_64 = model.evaluate(X_test, y_test)
print(f'Modelo 1 Camada (64 neurônios) -> Acurácia no Teste: {res_64[1] * 100:.2f}%')


## 25. O Efeito do Gargalo de Informação (32 Neurônios)

**O que estamos fazendo**: Treinar um modelo onde a camada oculta tem apenas 32 neurônios (menor que o número de classes de saída, 46).

**Por que fazer assim**: Demonstração prática de um dos conceitos mais importantes de Deep Learning: o gargalo de informação (*information bottleneck*).

> [!WARNING]
> Como 32 neurônios precisam codificar a informação necessária para 46 classes, ocorre perda irreparável de dados, derrubando o desempenho do modelo!



In [ ]:
# Arquitetura com gargalo de informação: apenas 32 neurônios na camada oculta para 46 classes
model = keras.Sequential([
  layers.Dense(32, activation='relu'),
  layers.Dense(46, activation='softmax')
])
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(partial_X_train,
                    partial_y_train,
                    epochs=20,
                    batch_size=512,
                    validation_data=(X_val, y_val),
                    verbose=0)
res_32 = model.evaluate(X_test, y_test)
print(f'Modelo com Gargalo (32 neurônios) -> Acurácia no Teste: {res_32[1] * 100:.2f}%')


## 26. Transição para o PyTorch: Modelo Sequencial & Loop de Treino Explícito

Agora, reimplementamos exatamente o mesmo classificador multiclasse utilizando o **PyTorch**.

> [!NOTE]
> **Diferença Fundamental de Paradigma:**
> No Keras, o método `.fit()` automatiza o loop de épocas e minibatches internamente. No PyTorch, temos o **controle total e explícito** de cada passo do cálculo:
> 1. `optimizer.zero_grad()`: Zera os gradientes acumulados da iteração anterior.
> 2. `outputs = model(X_batch)`: Executa o *Forward Pass* computacional.
> 3. `loss = criterion(outputs, y_batch)`: Calcula o custo da perda.
> 4. `loss.backward()`: Realiza a retropropagação do erro (*Backpropagation*) calculando os gradientes analíticos.
> 5. `optimizer.step()`: Atualiza os pesos aplicando o algoritmo RMSprop.


In [ ]:
# Importa módulos fundamentais do PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import accuracy_score

# Constrói a arquitetura sequencial equivalente no PyTorch
model = nn.Sequential(
  nn.Linear(10000, 32), # Camada totalmente conectada (10.000 entradas -> 32 ocultos)
  nn.ReLU(),            # Ativação não-linear ReLU
  nn.Linear(32, 46),    # Camada de saída (32 ocultos -> 46 classes)
  nn.Softmax(dim=1)     # Softmax aplicada ao longo da dimensão das classes
)

# Define a função de perda (CrossEntropyLoss) e o otimizador RMSprop
criterion = nn.CrossEntropyLoss()
optimizer = optim.RMSprop(model.parameters())

# Converte as matrizes NumPy em tensores PyTorch do tipo float32
partial_X_train_t = torch.tensor(partial_X_train, dtype=torch.float32)
partial_y_train_t = torch.tensor(partial_y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

# Hiperparâmetros de treinamento
num_epochs = 100
batch_size = 512

# Loop de épocas explícito no PyTorch
for epoch in range(num_epochs):
  # Itera sobre minibatches de tamanho 512
  for i in range(0, len(partial_X_train_t), batch_size):
    X_batch = partial_X_train_t[i:i+batch_size]
    y_batch = partial_y_train_t[i:i+batch_size]
    
    # 1. Zera os gradientes acumulados no grafo
    optimizer.zero_grad()
    # 2. Forward pass: calcula as predições do modelo
    outputs = model(X_batch)
    # 3. Calcula o custo
    loss = criterion(outputs, y_batch)
    # 4. Backward pass: calcula gradientes via regra da cadeia
    loss.backward()
    # 5. Atualiza os pesos com RMSprop
    optimizer.step()
    
  # Monitora a perda de validação a cada 10 épocas
  if (epoch + 1) % 10 == 0 or epoch == 0:
    val_loss = criterion(model(X_val_t), y_val_t)
    print(f'Época [{epoch+1:03d}/{num_epochs:03d}] | Perda Treino: {loss.item():.4f} | Perda Validação: {val_loss.item():.4f}')


## 27. Avaliação no PyTorch com `torch.no_grad()`

**O que estamos fazendo**: Avaliar a acurácia final do modelo PyTorch no conjunto de teste sem computar gradientes.

**Por que fazer assim**: O bloco `with torch.no_grad():` desativa a construção do grafo de autodiferenciação, economizando memória e acelerando a inferência.

> [!TIP]
> Utilizamos `torch.max(outputs, 1)` para encontrar a classe de maior probabilidade, equivalente ao `argmax` do NumPy!



In [ ]:
# Desativa o cálculo de gradientes durante a inferência no teste
with torch.no_grad():
  # Forward pass nas amostras de teste
  outputs = model(X_test_t)
  # Obtém o índice da maior probabilidade predita ao longo do eixo das classes (dim=1)
  _, predicted = torch.max(outputs, 1)
  # Calcula a acurácia comparando com os rótulos originais
  acc = accuracy_score(test_labels, predicted.numpy())
  print(f'Acurácia no Teste (PyTorch): {acc * 100:.2f}%')


---

## 🎓 Resumo da Aula & O Que Você Aprendeu

Nesta aula, realizamos um estudo completo de ponta a ponta sobre classificação multiclasse de textos jornalísticos com Deep Learning:

1. **Representação Textual**: Aprendemos como o texto é transformado em sequências de inteiros e depois em tensores binários esparsos (*Bag of Words* de 10.000 dimensões), tanto manualmente com NumPy quanto com o `Tokenizer` do Keras.
2. **Classificação em 46 Classes**: Configuramos a camada de saída com 46 neurônios ativados por `Softmax` e a perda `Categorical Crossentropy` associada a rótulos `to_categorical`.
3. **Diagnóstico Visual de Overfitting**: Vimos na prática como as curvas de perda de treino e validação divergem, indicando o momento exato de interromper o treinamento (em torno da 10ª época).
4. **Gargalo de Informação**: Provamos experimentalmente que quando as camadas ocultas possuem menos neurônios (32) que o número de classes de saída (46), o modelo sofre degradação severa de acurácia por perda irreversível de informação.
5. **Dualidade Keras vs PyTorch**: Treinamos a mesma arquitetura nos dois frameworks mais usados do mundo, dominando tanto a produtividade declarativa do Keras quanto a transparência e controle procedural do PyTorch.

---

## 👉 Próximo Passo
No **Lote 7 (Aulas 07a a 07d)**, ingressaremos no fascinante campo da **Visão Computacional**, comparando lado a lado redes totalmente conectadas (MLP Flat) versus **Redes Neurais Convolucionais (CNNs)** sobre o dataset MNIST, tanto em Keras quanto em PyTorch!
